In [1]:
import numpy as np
import sklearn
import pandas as pd
import matplotlib
import seaborn as sns

# 각 라이브러리 버전 출력
print("Numpy version:", np.__version__)
print("Scikit-learn version:", sklearn.__version__)
print("Pandas version:", pd.__version__)
print("Matplotlib version:", matplotlib.__version__)
print("Seaborn version:", sns.__version__)

Numpy version: 1.26.4
Scikit-learn version: 1.4.2
Pandas version: 2.2.2
Matplotlib version: 3.8.4
Seaborn version: 0.13.2


In [2]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import  OrdinalEncoder
from sklearn.ensemble import ExtraTreesClassifier

In [3]:
import sys
import os
import pandas as pd

# 현재 작업 디렉토리 경로를 가져와 shared codes 폴더의 위치를 sys.path에 추가합니다.
# sys.path에 추가된 경로에 있는 py 폴더는 임포트할 수 있다.
current_dir = os.getcwd()
shared_codes_dir = os.path.join(current_dir, '../shared codes')
sys.path.append(shared_codes_dir)


# cover_nan 모듈을 임포트
from cover_nan_0215_dahun import missing_value_removal_function

# 원본 train 데이터 로드
train = pd.read_csv("../shared codes/data/train.csv")
test = pd.read_csv("../shared codes/data/test.csv")

# missing_value_removal_function 사용
train_young, train_middle, train_old, train_unknown = missing_value_removal_function(train)
test_young, test_middle, test_old, test_unknown = missing_value_removal_function(test)

✅ '대리모 여부' 결측값을 최빈값 (0.0) 으로 대체 완료!
✅ 컬럼 삭제 완료: ['PGD 시술 여부', 'PGS 시술 여부', '난자 해동 경과일', '배아 해동 경과일']
✅ '난자 채취 경과일' 결측값을 중앙값 (0.0) 으로 대체 완료!
✅ '난자 혼합 경과일' 결측값을 중앙값 (0.0) 으로 대체 완료!
✅ '배아 이식 경과일' 결측값을 중앙값 (3.0) 으로 대체 완료!
✅ '대리모 여부' 결측값을 최빈값 (0.0) 으로 대체 완료!
✅ 컬럼 삭제 완료: ['PGD 시술 여부', 'PGS 시술 여부', '난자 해동 경과일', '배아 해동 경과일']
✅ '난자 채취 경과일' 결측값을 중앙값 (0.0) 으로 대체 완료!
✅ '난자 혼합 경과일' 결측값을 중앙값 (0.0) 으로 대체 완료!
✅ '배아 이식 경과일' 결측값을 중앙값 (3.0) 으로 대체 완료!


In [4]:
def data_preprocessing(train, test):
    # 미리 ID 저장

    index_test = test['ID'].copy() 

    # Drop ID and target columns
    X = train.drop(['임신 성공 여부', 'ID'], axis=1)
    y = train['임신 성공 여부']
    test = test.drop('ID', axis=1)

    # Categorical columns
    categorical_columns = X.select_dtypes(include=['object', 'category']).columns.tolist()

    # Encoding
    from sklearn.preprocessing import OrdinalEncoder
    ordinal_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

    X_train_encoded = X.copy()
    X_train_encoded[categorical_columns] = ordinal_encoder.fit_transform(X[categorical_columns])

    X_test_encoded = test.copy()
    X_test_encoded[categorical_columns] = ordinal_encoder.transform(test[categorical_columns])

    # Drop unnecessary columns
    columns_to_drop = [
        "남성 주 불임 원인",
        "남성 부 불임 원인",
        "불임 원인 - 정자 농도",
        "불임 원인 - 정자 면역학적 요인",
        "불임 원인 - 정자 운동성",
        "불임 원인 - 정자 형태",
        '배란 유도 유형'
    ]
    X_train_encoded = X_train_encoded.drop(columns=columns_to_drop)    
    X_test_encoded = X_test_encoded.drop(columns=columns_to_drop)

    # ID 다시 추가

    X_test_encoded['ID'] = index_test.values

    return X_train_encoded, X_test_encoded, y


In [5]:
X_train_encoded_young, X_test_encoded_young, y_young = data_preprocessing(train_young, test_young)
X_train_encoded_middle, X_test_encoded_middle, y_middle = data_preprocessing(train_middle, test_middle)
X_train_encoded_old, X_test_encoded_old, y_old = data_preprocessing(train_old, test_old)
X_train_encoded_unknown, X_test_encoded_unknown, y_unknown = data_preprocessing(train_unknown, test_unknown)

In [6]:
X_train_encoded_young = pd.concat([X_train_encoded_young, y_young], axis=1)
X_train_encoded_middle = pd.concat([X_train_encoded_middle, y_middle], axis=1)
X_train_encoded_old = pd.concat([X_train_encoded_old, y_old], axis=1)
X_train_encoded_unknown = pd.concat([X_train_encoded_unknown, y_unknown], axis=1)

test_young_id = X_test_encoded_young["ID"]
X_test_encoded_young= X_test_encoded_young.drop(columns=["ID"])

test_middle_id = X_test_encoded_middle["ID"]
X_test_encoded_middle= X_test_encoded_middle.drop(columns=["ID"])

test_old_id = X_test_encoded_old["ID"]
X_test_encoded_old= X_test_encoded_old.drop(columns=["ID"])

test_unknown_id = X_test_encoded_unknown["ID"]
X_test_encoded_unknown= X_test_encoded_unknown.drop(columns=["ID"])

In [7]:
import pandas as pd
from autogluon.tabular import TabularPredictor

def train_model(training_data, target_variable, config):
    """
    주어진 데이터와 하이퍼파라미터 설정을 사용하여 모델을 학습하고,
    학습된 TabularPredictor 객체를 반환합니다.
    """
    model = TabularPredictor(label=target_variable, eval_metric="roc_auc")
    model.fit(
        training_data,
        presets="best_quality",
        num_bag_folds=10,
        hyperparameters=config,
        num_stack_levels=1
    )
    return model


def save_submission(model, test_data, id_series, output_path):
    """
    테스트 데이터에 대해 확률 예측을 수행하고, 제출 파일을 생성하여 저장합니다.
    """
    # 예측 수행 (양성 클래스의 확률 사용)
    prediction_probs = model.predict_proba(test_data)
    submission_data = pd.DataFrame({
        "ID": id_series,
        "probability": prediction_probs[1]
    })
    submission_data.to_csv(output_path, index=False)
    print(f"Submission 생성 : {output_path}")
    return submission_data


def combine_submissions(submission_young, submission_middle, submission_old, submission_unknown, output_path="./Result/Submission_combined.csv"):
    """
    세 개의 제출 파일을 합치고 저장합니다.
    """
    combined_submission = pd.concat([submission_young, submission_middle, submission_old, submission_unknown]).reset_index(drop=True)
    combined_submission.to_csv(output_path, index=False)
    print(f"Combined submission 생성 : {output_path}")


def main():
    # 하이퍼파라미터 설정 (각 모델에 대해 기본 설정)
    hyperparams = {
        "GBM": {},
        "CAT": {},
        "XGB": {}
    }
    
    target_col = "임신 성공 여부"
    
    # Young 모델 학습 및 예측
    trained_model_young = train_model(X_train_encoded_young, target_col, hyperparams)
    submission_young = save_submission(trained_model_young, X_test_encoded_young, test_young_id, output_path="./Result/Submission_young.csv")
    
    # Middle 모델 학습 및 예측
    trained_model_middle = train_model(X_train_encoded_middle, target_col, hyperparams)
    submission_middle = save_submission(trained_model_middle, X_test_encoded_middle, test_middle_id, output_path="./Result/Submission_middle.csv")
    
    # Old 모델 학습 및 예측
    trained_model_old = train_model(X_train_encoded_old, target_col, hyperparams)
    submission_old = save_submission(trained_model_old, X_test_encoded_old, test_old_id, output_path="./Result/Submission_old.csv")

    # Old 모델 학습 및 예측
    trained_model_unknown = train_model(X_train_encoded_unknown, target_col, hyperparams)
    submission_unknown = save_submission(trained_model_unknown, X_test_encoded_unknown, test_unknown_id, output_path="./Result/Submission_unknown.csv")

    # 세 개의 제출 파일 합치기
    combine_submissions(submission_young, submission_middle, submission_old, submission_unknown)


if __name__ == "__main__":
    main()


No path specified. Models will be saved in: "AutogluonModels\ag-20250222_113856"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.2
Python Version:     3.12.3
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.22621
CPU Count:          16
Memory Avail:       17.45 GB / 31.93 GB (54.7%)
Disk Space Avail:   196.92 GB / 930.76 GB (21.2%)
Presets specified: ['best_quality']
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=10, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked overfitting and enable or disable stacking as a consequence.
	This is used to identify the optimal `num_stack_levels` value. Copies of AutoGluon will be fit on subsets of the data. Then holdou

Submission 생성 : ./Result/Submission_young.csv


Leaderboard on holdout data (DyStack):
                 model  score_holdout  score_val eval_metric  pred_time_test  pred_time_val   fit_time  pred_time_test_marginal  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0  WeightedEnsemble_L2       0.703764   0.704314     roc_auc        2.785161       0.746462  36.894469                 0.011999                0.008020           0.654956            2       True          4
1  WeightedEnsemble_L3       0.703764   0.704314     roc_auc        2.786161       0.746446  37.452657                 0.012998                0.008004           1.213145            3       True          8
2      CatBoost_BAG_L1       0.703722   0.703926     roc_auc        0.670714       0.035519  25.574478                 0.670714                0.035519          25.574478            1       True          2
3      CatBoost_BAG_L2       0.703012   0.703123     roc_auc        2.908657       0.774449  50.205524                 0.135494          

Submission 생성 : ./Result/Submission_middle.csv


Leaderboard on holdout data (DyStack):
                 model  score_holdout  score_val eval_metric  pred_time_test  pred_time_val   fit_time  pred_time_test_marginal  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0      CatBoost_BAG_L1       0.727876   0.740744     roc_auc        0.619188       0.030022   9.482584                 0.619188                0.030022           9.482584            1       True          2
1       XGBoost_BAG_L2       0.727767   0.734182     roc_auc        2.794510       0.457574  22.044759                 0.314649                0.149511           4.121455            2       True          7
2  WeightedEnsemble_L2       0.727416   0.741214     roc_auc        0.989803       0.165186  14.557942                 0.008512                0.005000           0.386932            2       True          4
3  WeightedEnsemble_L3       0.727416   0.741214     roc_auc        0.991289       0.164186  14.849558                 0.009997          

Submission 생성 : ./Result/Submission_old.csv


Leaderboard on holdout data (DyStack):
                 model  score_holdout  score_val eval_metric  pred_time_test  pred_time_val   fit_time  pred_time_test_marginal  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0      LightGBM_BAG_L2       0.868852   0.887265     roc_auc        2.658625       0.054049  11.250364                 0.100510                0.009998           2.850390            2       True          5
1      LightGBM_BAG_L1       0.859745   0.828851     roc_auc        1.646796       0.006001   3.042214                 1.646796                0.006001           3.042214            1       True          1
2  WeightedEnsemble_L3       0.848816   0.896998     roc_auc        3.025772       0.093067  17.939353                 0.009000                0.000000           0.021516            3       True          8
3       XGBoost_BAG_L1       0.843352   0.813872     roc_auc        0.321071       0.023050   2.254647                 0.321071          

Submission 생성 : ./Result/Submission_unknown.csv
Combined submission 생성 : ./Result/Submission_combined.csv
